# Baseline: K-Means on Raw Graph Features

This notebook trains a K-Means clustering model directly on the **raw root-node features**
extracted from the pre-built dependency graph dataset — without passing them through a GNN.

It serves as a graph-aware baseline to compare against the GCL-trained GNN model in
`train_gnn_model.ipynb`. Both notebooks use the same graph dataset, the same train/test
split seed, and the same evaluation metrics, so results are directly comparable.

## Workflow
1. Load the cached PyG graph dataset from `data/gnn_graph_dataset.pkl`
2. Extract the **root-node raw feature vector** (`data.x[0]`, dim=27) for each package
3. Apply the same train/test split as the GNN notebook (80/20, seed=42)
4. Standardise features and fit K-Means on the **test set** using the Elbow Method
5. Evaluate: risk score per cluster → top-K% clusters → Recall / Precision / F1

## Setup

In [26]:
import sys
import os
import pickle
import random

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import joblib

# ── Paths ──────────────────────────────────────────────────────────────────────────
GRAPH_CACHE   = '../data/gnn_graph_dataset.pkl'
MODEL_OUTPUT  = '../models/lib/raw_graph_kmeans.pkl'
SCALER_OUTPUT = '../models/lib/raw_graph_scaler.pkl'

# ── Hyper-parameters (match train_gnn_model.ipynb) ─────────────────────────────────────
TRAIN_RATIO = 0.8
TOP_K_PCT   = 0.10

os.makedirs('../models/lib', exist_ok=True)
print('Setup complete.')

Setup complete.


## Step 1: Load graph dataset

In [27]:
with open(GRAPH_CACHE, 'rb') as f:
    graph_dataset = pickle.load(f)

# Drop any graphs with empty node feature matrices (defensive)
before = len(graph_dataset)
graph_dataset = [d for d in graph_dataset if d['data'].x.size(0) > 0]
n_empty = before - len(graph_dataset)

n_vuln = sum(1 for d in graph_dataset if d['label'] == 1)
print(f'Loaded {len(graph_dataset)} graphs  ({n_vuln} vulnerable)'
      + (f'  [{n_empty} empty graphs dropped]' if n_empty else ''))

Loaded 9668 graphs  (4823 vulnerable)


## Step 2: Extract raw root-node features

`build_pyg_data` guarantees the seed package is always placed at node index 0.
So `data.x[0]` gives its 27-dimensional raw feature vector directly, with no GNN involved.

In [28]:
features = []
labels   = []
pkg_ids  = []

for item in tqdm(graph_dataset, desc='Extracting root features'):
    features.append(item['data'].x[0].numpy())   # root node raw features (27-dim)
    labels.append(item['label'])
    pkg_ids.append(item['pkg_id'])

X_raw  = np.array(features) # (N, 27)
labels = np.array(labels)

print(f'Feature matrix : {X_raw.shape}')
print(f'Vulnerable     : {labels.sum()} / {len(labels)}')

Extracting root features: 100%|██████████| 9668/9668 [00:00<00:00, 528393.87it/s]

Feature matrix : (9668, 27)
Vulnerable     : 4823 / 9668


## Step 3: Train / test split

Uses the same seed and ratio as `train_gnn_model.ipynb` so the test sets are identical.

In [29]:
indices = list(range(len(graph_dataset)))
random.seed(42)
random.shuffle(indices)

split_idx    = int(len(indices) * TRAIN_RATIO)
test_indices = indices[split_idx:]

X_test      = X_raw[test_indices]
test_labels = labels[test_indices]
test_pkgids = [pkg_ids[i] for i in test_indices]

print(f'Test  : {len(X_test)}')
print(f'Vulnerable in test : {test_labels.sum()} / {len(test_labels)}')

Test  : 1934
Vulnerable in test : 989 / 1934


## Step 4: Standardise and K-Means — Elbow Method

In [30]:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_test)

K_RANGE  = range(2, 21)
inertias = {}

for k in tqdm(K_RANGE, desc='Elbow sweep'):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias[k] = km.inertia_

elbow_df = pd.DataFrame({
    'K'      : list(inertias.keys()),
    'Inertia': list(inertias.values()),
})
elbow_df['Delta']    = elbow_df['Inertia'].diff().abs()
elbow_df['Delta2']   = elbow_df['Delta'].diff().abs()
elbow_df['Gain_pct'] = (elbow_df['Delta'] / elbow_df['Inertia'].shift(1) * 100).round(2)

print(elbow_df.to_string(index=False))

best_k = int(elbow_df.loc[elbow_df['Delta2'].idxmax(), 'K'])
print(f'\nAuto-detected elbow K = {best_k}')

Elbow sweep: 100%|██████████| 19/19 [00:00<00:00, 21.40it/s]

 K      Inertia       Delta      Delta2  Gain_pct
 2 45630.906250         NaN         NaN       NaN
 3 41086.562500 4544.343750         NaN      9.96
 4 38080.445312 3006.117188 1538.226562      7.32
 5 35003.441406 3077.003906   70.886719      8.08
 6 34435.710938  567.730469 2509.273438      1.62
 7 31417.351562 3018.359375 2450.628906      8.77
 8 29694.451172 1722.900391 1295.458984      5.48
 9 27974.148438 1720.302734    2.597656      5.79
10 26629.753906 1344.394531  375.908203      4.81
11 25499.326172 1130.427734  213.966797      4.24
12 23813.019531 1686.306641  555.878906      6.61
13 23175.927734  637.091797 1049.214844      2.68
14 22041.531250 1134.396484  497.304688      4.89
15 21369.478516  672.052734  462.343750      3.05
16 20010.015625 1359.462891  687.410156      6.36
17 19259.330078  750.685547  608.777344      3.75
18 18739.953125  519.376953  231.308594      2.70
19 17993.875000  746.078125  226.701172      3.98
20 17226.037109  767.837891   21.759766      4.27


In [31]:
# Override best_k here if you disagree with the auto-detection
# best_k = 10

final_km       = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = final_km.fit_predict(X_scaled)

results_df = pd.DataFrame({
    'pkg_id' : test_pkgids,
    'label'  : test_labels,
    'cluster': cluster_labels,
})

print(f'Trained KMeans with K={best_k}')
print(results_df.groupby('cluster')['label'].agg(['sum', 'count']).rename(
    columns={'sum': 'n_vulnerable', 'count': 'cluster_size'}
))

Trained KMeans with K=6
         n_vulnerable  cluster_size
cluster                            
0                 161           447
1                 184           192
2                  18           259
3                 162           558
4                 464           477
5                   0             1


In [32]:
from sklearn.metrics import silhouette_score

sil = silhouette_score(X_scaled, cluster_labels)
print(f'Silhouette score (K={best_k}, raw graph features): {sil:.4f}')
print('  > 0.5   → strong structure')
print('  0.2–0.5 → moderate structure')
print('  < 0.2   → weak / overlapping clusters')

Silhouette score (K=6, raw graph features): 0.1046
  > 0.5   → strong structure
  0.2–0.5 → moderate structure
  < 0.2   → weak / overlapping clusters


## Step 5: Evaluate

**Risk score** per cluster = `n_vulnerable / cluster_size`  
**Top-K clusters** = top 10% of clusters ranked by risk score

Metrics:
- **Recall** — fraction of all known-vulnerable packages that fall in the top-K clusters
- **Precision** — fraction of packages inside the top-K clusters that are actually vulnerable

In [33]:
cluster_stats = (
    results_df.groupby('cluster')['label']
    .agg(n_vulnerable='sum', cluster_size='count')
    .assign(risk_score=lambda df: df['n_vulnerable'] / df['cluster_size'])
    .sort_values('risk_score', ascending=False)
    .reset_index()
)

print('=== All clusters ranked by risk score ===')
print(cluster_stats.to_string(index=False))

# Top-K% of PACKAGES: add clusters in decreasing risk order until ~K% of packages are flagged
n_top_pkgs = max(1, round(len(results_df) * TOP_K_PCT))
top_clusters = set()
n_flagged = 0
for _, row in cluster_stats.iterrows():
    top_clusters.add(int(row['cluster']))
    n_flagged += int(row['cluster_size'])
    if n_flagged >= n_top_pkgs:
        break
print(f'\nTop clusters covering ~{TOP_K_PCT*100:.0f}% of packages ({n_flagged} / {len(results_df)}): {top_clusters}')

total_vulnerable = (results_df['label'] == 1).sum()
in_top_mask      = results_df['cluster'].isin(top_clusters)
true_positives   = ((results_df['label'] == 1) & in_top_mask).sum()
top_cluster_size = in_top_mask.sum()

recall    = true_positives / total_vulnerable if total_vulnerable else 0.0
precision = true_positives / top_cluster_size if top_cluster_size  else 0.0
f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

print(f'\n--- Evaluation (top ~{TOP_K_PCT*100:.0f}% packages as "risky") ---')
print(f'Total packages     : {len(results_df)}')
print(f'Total vulnerable   : {total_vulnerable}')
print(f'Flagged packages   : {top_cluster_size}')
print(f'True positives     : {true_positives}')
print(f'Recall             : {recall:.4f}  ({recall*100:.1f}%)')
print(f'Precision          : {precision:.4f}  ({precision*100:.1f}%)')
print(f'F1                 : {f1:.4f}')

=== All clusters ranked by risk score ===
 cluster  n_vulnerable  cluster_size  risk_score
       4           464           477    0.972746
       1           184           192    0.958333
       0           161           447    0.360179
       3           162           558    0.290323
       2            18           259    0.069498
       5             0             1    0.000000

Top clusters covering ~10% of packages (477 / 1934): {4}

--- Evaluation (top ~10% packages as "risky") ---
Total packages     : 1934
Total vulnerable   : 989
Flagged packages   : 477
True positives     : 464
Recall             : 0.4692  (46.9%)
Precision          : 0.9727  (97.3%)
F1                 : 0.6330


## Step 6: Save model

In [34]:
joblib.dump(final_km, MODEL_OUTPUT)
joblib.dump(scaler,   SCALER_OUTPUT)

cluster_stats.to_csv('../models/lib/raw_graph_cluster_risk_scores.csv', index=False)

print(f'KMeans saved      → {MODEL_OUTPUT}')
print(f'Scaler saved      → {SCALER_OUTPUT}')
print(f'Risk scores saved → ../models/lib/raw_graph_cluster_risk_scores.csv')

KMeans saved      → ../models/lib/raw_graph_kmeans.pkl
Scaler saved      → ../models/lib/raw_graph_scaler.pkl
Risk scores saved → ../models/lib/raw_graph_cluster_risk_scores.csv
